Quick and dirty notebook for simulation of coherent scattering data of magnetic samples in fraunhofer far-field regime

# Import

In [ ]:
# Import general libraries
import sys, os
from os.path import join, split
from importlib import reload
from copy import deepcopy
from tqdm.auto import tqdm
import numpy as np
# scipy
import scipy
# plotting
import matplotlib.pyplot as plt
# Interactive plotting
import ipywidgets
import time
%matplotlib widget
plt.rcParams["figure.constrained_layout.use"] = True

In [ ]:
# Imports from our own codebase
from scattering_calculator.experimental_conditions import detector, light_beam
from scattering_calculator.sample_generator import pattern_generator
from scattering_calculator.sample_generator import structures
from scattering_calculator.beam_propagator import Jones_propagator
from scattering_calculator.utils import masking
from scattering_calculator.interactive.interactive_widgets import cimshow
from scattering_calculator.simulation_pipelines import simulate_experiment, simulation_configuration


### EXPERIMENTAL GEOMETRY

In [ ]:
# ===================
# X-ray Source
# ===================

polarization = "CR"
x_ray_energy = 787.9  # eV
x_ray_photon_flux = 1e10  # Photons per pulse
coherence_length = 1e-6  # in m, optional for now, will be used in future to simulate partial coherence effects

#######################################
# Create config class for x-ray source
xrayconfig = simulation_configuration.XRayConfig(
    energy=x_ray_energy,
    pol=polarization,
    photon_flux=x_ray_photon_flux,
    coherence_length=coherence_length,
)
xrayconfig.setup()

In [ ]:
# ==================
# DETECTOR-BEAMSTOP GEOMETRY
# ==================
detector_pixel_size = 20e-6  # in m
detector_pixel_shape = (1300, 1300)
detector_distance = 0.02  # in m
detector_center = (650, 650)  # in px


# Optional: Define a beamstop
# Basic parameters for beamstop
beamstop_distance = 0.001  # in m
beamstop_center = np.array(detector_pixel_shape) // 2  # in px

# Select beamstop method and parameters
beamstop_method = "circular"  # "circular", "rectangular", None
beamstop_radius = 0.5e-3  # in m


####################################
# Create config class for beamstop
beamstop_config = simulation_configuration.BeamstopConfig(
    bs_method=beamstop_method,
    bs_detector_distance=beamstop_distance,
    bs_center=beamstop_center,
    bs_config={"radius": beamstop_radius},
)

# Create config class for detector
detectorconfig = simulation_configuration.DetectorConfig(
    pixel_size=detector_pixel_size,
    shape=detector_pixel_shape,
    sample_to_detector_distance=detector_distance,
    detector_center=detector_center,
    beamstop_config=beamstop_config
    
)
detectorconfig.setup()

# SAMPLE 

In [ ]:
#  ===================
# BASIC SAMPLE DIMENSION PARAMETERS
# ===================

## This is the resolution we will have thanks to the detector
real_space_pixel_size = (
    detectorconfig.calc_realspace_resolution(xrayconfig.beam_params) / 2
)

# Do I actually need this?
simulationconfig = simulation_configuration.SimulationConfig(shape = detectorconfig.shape, real_space_pixel_size=real_space_pixel_size)
simulationconfig.setup()

### SAMPLE STACK STRUCTURE AND OPTICAL PROPERTIES

In [ ]:
# ===================
# MATERIAL RECIPE
# ===================
recipe = "Au(700)/Cr(300)/SiN(200)/Co(90)/Pt(120)/Al(60)"
# "Au(1000)/SiN(200)/Ta(5)/[Pt(1)/Co(1)]x15/Pt(3)"


sample_shape = np.array(
    [
        0,
        2 * detectorconfig.shape[0],
        2 * detectorconfig.shape[1],
    ]  # why is there a factor of 2?
,dtype=int)  # in pixels

sampleconfig = simulation_configuration.SampleConfig(
    recipe=recipe,
    sample_shape=sample_shape,
    real_space_pixel_size=real_space_pixel_size,
    xray_config=xrayconfig,
    sample_name="Test_Sample",
)
sampleconfig.setup()

### - magnetic domains

In [ ]:
stripe_width = 20e-9
sigma = 1e-9
angle_stripes = np.pi / 4
wave_amplitudes = 20e-9
wave_scale = 20e-9

magnetic_pattern_config = simulation_configuration.MagneticPatternConfig(
    pattern_type_method="wavy_stripe_pattern",
    shape=sample_shape[1:],
    real_space_pixel_size=real_space_pixel_size,
    pattern_config_length={
        "stripe_width": stripe_width,
        "sigma": sigma,
        "waviness_amplitude": wave_amplitudes,
        "waviness_scale": wave_scale,
    },
    pattern_config={
        "angle_stripes": angle_stripes,
    },
)
magnetic_pattern_config.create_pattern()
magnetic_pattern_config.plot_pattern()

In [ ]:
%time
# Skyrmion and Screening diameter
skyrmion_radius = 3e-9  # m
screening_radius = (
    1.5 * skyrmion_radius
)  # m, either only even or odd, otherwise script will fail
skyrmion_smoothing = 1

# Number of skyrmions
# If this number is too high, the script may take forever ...
# (brute force algorithm)
number_of_skyrmions = 60000
max_nr_iteration = 100000  # 10 * sample["no_skyr"]

magnetic_pattern_config = simulation_configuration.MagneticPatternConfig(
    pattern_type_method="skyrmion_pattern",
    shape=sample_shape[1:],
    real_space_pixel_size=real_space_pixel_size,
    pattern_config_length={
        "skyr_radius": skyrmion_radius,
        "screening_radius": screening_radius
    },
    pattern_config={
        "number_skyr": number_of_skyrmions,
        "number_iter": max_nr_iteration,
        "sigma": skyrmion_smoothing
    }
)
magnetic_pattern_config.create_pattern()
magnetic_pattern_config.plot_pattern()

In [ ]:
magnetic_pattern = magnetic_pattern_config.magnetic_pattern
magnetization = pattern_generator.map_magnetization_to_3d(
    np.zeros_like(magnetic_pattern),
    np.sqrt(1 - np.abs(magnetic_pattern) ** 2),
    magnetic_pattern,
    nr_repeats=sample_shape[0],
)

sampleconfig.assign_magnetic_pattern(magnetization)

# - holography mask

In [ ]:
sampleconfig.sample_structure

In [ ]:
front_aperture.visualize_aperture()

In [ ]:
#  ===================
# HOLOGRAPHY MASK DESIGN
# ===================

apertures_radius = [60e-9, 6e-9, 4e-9]
apertures_types = ["OH", "RH", "RH"]
apertures_centers = [(0, 0), (0.2e-6, -0.15e-6), (0.15e-6, 0.15e-6)]
apertures_sigma = [1e-9, 0.1e-9, 0.1e-9]

front_aperture_config = simulation_configuration.FrontApertureConfig(
    aperture_method="FTH_circular",
    aperture_shape=sample_shape,
    real_space_pixel_size=sampleconfig.sample_structure.real_space_pixel_size,
    aperture_thickness=sum(sampleconfig.sample_structure.layer_thicknesses),
    aperture_config=dict(
        apertures_type=apertures_types,
        apertures_radius=apertures_radius,
        apertures_center=apertures_centers,
        apertures_sigma=apertures_sigma,
        thickness_OH=np.sum(
            sampleconfig.sample_structure.layer_thicknesses[: sampleconfig.sample_structure.layer_names.index("SiN")]
        ),
    ),
)
front_aperture_config.setup()
front_aperture_config.visualize_aperture()

aperture_mask = front_aperture_config.return_aperture()
sampleconfig.assign_aperture_mask(aperture_mask)

In [ ]:
sampleconfig.sample_structure.calculate_final_dielectric_tensor()

### HOLOGRAM COMPUTATION

In [ ]:
# Params for gaussian beam
illumination_function = "gaussian"
illumination_center = (0, 0)
illumination_focus_distance = 1e-3  # in m
illumination_fwhm = 0.5e-6  # in m, it is not showing correct fwhm?

illuminationconfig = simulation_configuration.IlluminationConfig(
    XRayConfig = xrayconfig,
    shape = sample_shape[1:],
    real_space_pixel_size = real_space_pixel_size,
    illumination_function = illumination_function,
    illumination_config = {
        "center": illumination_center,
        "distance": illumination_focus_distance,
        "fwhm": illumination_fwhm,
    },
)
illuminationconfig.setup()
illuminationconfig.visualize_illumination()

In [ ]:
time0 = time.time()
# compute dielectric tensor

print("dielectric tensor   - %0.1f s" % (time.time() - time0))
time0 = time.time()

holos = np.zeros((2, *detectorconfig.shape))

for i, polarization in enumerate(["CR", "CL"]):
    illuminationconfig.update_polarization(polarization)

    print("illumination wave   - %0.1f s"%(time.time()-time0))
    time0=time.time()

    samplepropagationconfig = simulation_configuration.SamplePropagatorConfig(
        SampleConfig = sampleconfig,
        IlluminationConfig = illuminationconfig,
        propagator_method = "Jones",
        propagator_config = {},
        )
    samplepropagationconfig.setup()

    detectorconfig.assign_propagated_wavefront(samplepropagationconfig)
    detectorconfig.detect_hologram()
    detectorconfig.hologram_exp.gnomonic_projection()
    holos[i] = detectorconfig.return_detected_hologram()

In [ ]:
samplepropagationconfig.wavefront.exit_wave.shape

In [ ]:
# circ
amp = np.abs(
    samplepropagationconfig.wavefront.exit_wave[..., 0]**2
    + samplepropagationconfig.wavefront.exit_wave[..., 0]**2
)/np.sqrt(2)
phase = np.angle(samplepropagationconfig.wavefront.exit_wave[..., 0] )

# lin horizontal
amp = np.abs(samplepropagationconfig.wavefront.exit_wave[..., 0])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
im0 = ax[0].imshow(amp, cmap="inferno")
fig.colorbar(im0, ax=ax[0])
im1 = ax[1].imshow(phase, cmap="twilight")      
fig.colorbar(im1, ax=ax[1])

In [ ]:
cimshow(amp)

In [ ]:
cimshow(samplepropagationconfig.wavefront.exit_wave[..., 1])

In [ ]:
cimshow(holos)

In [ ]:
reload(simulation_configuration)
reload(detector)
reload(light_beam)
reload(structures)

In [ ]:
fth=Jones_propagator.reconstruct((holos[0]-holos[1]))#*scipy.ndimage.gaussian_filter(1.*scipy.ndimage.binary_erosion(1-beamstop.beamstop, iterations=25,border_value=1), sigma=10))
fth2=np.abs(fth)
fth2[fth.shape[0]//2:,:]=np.imag(fth)[fth.shape[0]//2:,:]
cimshow(fth2, cmap="gray")

In [ ]:
plt.close("all")


fig,ax=plt.subplots(2,7, figsize=(14,4))

roi=np.s_[900:1150,120:400]
roi2=np.s_[1:900,:1200]
roi3=np.s_[630:670,1135:1170]


fth=Jones_propagator.reconstruct((holos[0]+holos[1]))

ax[0,0].imshow(np.log10(holos[0]+holos[1]))

ax[0,1].imshow((np.real(fth)[roi]))
ax[0,2].imshow((np.imag(fth)[roi]))

ax[0,3].imshow((np.real(fth)[roi2]))
ax[0,4].imshow((np.imag(fth)[roi2]))

ax[0,5].imshow((np.real(fth)[roi3]))
ax[0,6].imshow((np.imag(fth)[roi3]))

fth=Jones_propagator.reconstruct(holos[0]-holos[1])

ax[1,0].imshow(np.log10(np.abs(holos[0]-holos[1])))

ax[1,1].imshow((np.real(fth)[roi]))
ax[1,2].imshow((np.imag(fth)[roi]))

ax[1,3].imshow((np.real(fth)[roi2]))
ax[1,4].imshow((np.imag(fth)[roi2]))

ax[1,5].imshow((np.real(fth)[roi3]))
ax[1,6].imshow((np.imag(fth)[roi3]))


In [ ]:
fth=Jones_propagator.reconstruct(holos[0]-holos[1])
fth2=np.real(fth)
fth2[fth.shape[0]//2:,:]=np.imag(fth)[fth.shape[0]//2:,:]
cimshow(fth2, cmap="gray")

In [ ]:

fig,ax=plt.subplots()
ax.imshow(Jones_propagator.E_I((wavefront.exit_wave[:,:,:])))

In [ ]:
cimshow(Jones_propagator.E_I((wavefront.exit_wave[:,:,:])))

In [ ]:

cimshow(np.imag(Jones_propagator.reconstruct(hologram_exp.hologram_exp)), cmap="gray")

In [ ]:
plt.close("all")


fig,ax=plt.subplots(2,7, figsize=(14,4))

roi=np.s_[270:460,300:490]
roi2=np.s_[280:460,820:990]
roi3=np.s_[630:670,1135:1170]


fth=Jones_propagator.reconstruct((holos[0]+holos[1]))

ax[0,0].imshow(np.log10(holos[0]+holos[1]))

ax[0,1].imshow((np.real(fth)[roi]))
ax[0,2].imshow((np.imag(fth)[roi]))

ax[0,3].imshow((np.real(fth)[roi2]))
ax[0,4].imshow((np.imag(fth)[roi2]))

ax[0,5].imshow((np.real(fth)[roi3]))
ax[0,6].imshow((np.imag(fth)[roi3]))

fth=Jones_propagator.reconstruct(holos[0]-holos[1])

ax[1,0].imshow(np.log10(np.abs(holos[0]-holos[1])))

ax[1,1].imshow((np.real(fth)[roi]))
ax[1,2].imshow((np.imag(fth)[roi]))

ax[1,3].imshow((np.real(fth)[roi2]))
ax[1,4].imshow((np.imag(fth)[roi2]))

ax[1,5].imshow((np.real(fth)[roi3]))
ax[1,6].imshow((np.imag(fth)[roi3]))


In [ ]:


reload(Jones_propagator)
reload(light_beam)
reload(detector)
reload(structures)


hologram_exp = detector.detector_hologram( exp_detector, wavefront.hologram, beam_params,sample.real_space_pixel_size, beamstop)
hologram_exp.gnomonic_projection()


In [ ]:
hologram_exp.add_noise()


In [ ]:

cimshow(hologram_exp.hologram_exp)